In [1]:
from pyfluids import Fluid, FluidsList, Input

import numpy as np 
T = 90 #K
T_C = T - 273.15 #Celsius

ethane = Fluid(FluidsList.Ethane)
# Set state for ethane
Temperature = 180
T_celsius = Temperature - 273.15  # Convert Kelvin to Celsius
ethane = ethane.with_state(
    Input.temperature(T_celsius),  # Temperature in Kelvin
    Input.pressure(101325)   # Pressure in Pascals (≈1 atm)
)

# Calculate properties
k_ethane = ethane.conductivity
rho_ethane = ethane.density
cp_ethane = ethane.specific_heat
surface_tension = 0.025 # N/m for ethane in air 
dynamic_viscosity = ethane.dynamic_viscosity
print(f"Ethane properties, 1 atm:")
print(f"Thermal conductivity: {k_ethane:.4f} W/(m*K)")
print(f"Density: {rho_ethane:.4f} kg/m^3")
print(f"Specific heat: {cp_ethane:.4f} J/(kg*K)")
print(f"Surface tension: {surface_tension:.4f} N/m")
print(f"Dynamic viscosity: {dynamic_viscosity:.4e} Pa*s")
thermal_diffusivity_ethane = k_ethane / (rho_ethane * cp_ethane)
print(f"Thermal diffusivity: {thermal_diffusivity_ethane:.4e} m^2/s")


Ethane properties, 1 atm:
Thermal conductivity: 0.1713 W/(m*K)
Density: 549.5297 kg/m^3
Specific heat: 2421.3569 J/(kg*K)
Surface tension: 0.0250 N/m
Dynamic viscosity: 1.7615e-04 Pa*s
Thermal diffusivity: 1.2877e-07 m^2/s


In [2]:
# Geometry
substrate = "cryochip" 
substrate_list = ["cryochip", "cryosilico", "autogrid", "si_old_chip", "cryoemgrid"]
reference_front_view_dimension = {
    "cryochip": 1e-3,  
    "cryosilico": 1.5e-3,
    "si_old_chip": 1.65e-3,
    "autogrid": 1.75e-3,
    "cryoemgrid": 1.5e-3
}

reference_side_view_dimension = {
    "cryochip": 50e-6,  # 20 microns
    "cryosilico": 100e-6, # 50 microns
    "si_old_chip": 150e-6, # 100 microns
    "autogrid": 150e-6, # 150 microns
    "cryoemgrid": 12.5e-6 # 50 microns
}

velocity = 1 # m/s


In [3]:
fluid_non_dimensional_numbers = ["bond_number", "capillary_number", "weber_number", "reynolds_number", "ohnesorge_number", "grasshoff_number", "froude_number", "froude_number_root"]
reynolds_number = lambda rho, velocity, length, viscosity: (rho * velocity * length) / viscosity
capillary_number = lambda surface_tension, velocity, viscosity: (viscosity * velocity) / surface_tension
bond_number = lambda rho, gravity, length, surface_tension: (rho * gravity * length**2) / surface_tension
weber_number = lambda rho, velocity, length, surface_tension: (rho * velocity**2 * length) / surface_tension
froude_number = lambda velocity, length, gravity: (velocity**2) / (length * gravity)
froude_number_root = lambda velocity, length, gravity: np.sqrt(velocity**2 / (length * gravity))
ohnesorge_number = lambda viscosity, rho, surface_tension, length: viscosity / np.sqrt(rho * surface_tension * length)
grashoff_number = lambda rho, gravity, length, viscosity, thermal_diffusivity: (rho * gravity * length**3) / (viscosity**2 * thermal_diffusivity)
capillary_length = lambda surface_tension, rho, gravity: np.sqrt(surface_tension / (rho * gravity))

In [4]:
# Calculate non-dimensional numbers for each substrate
numbers = {}
g = 9.81  # m/s^2, acceleration due to gravity

for substrate in substrate_list:
    numbers[substrate] = {
        "front" : {},
        "side" : {}
    }
    # Front view calculations
    length_front = reference_front_view_dimension[substrate]
    numbers[substrate]["front"]["bond_number"] = bond_number(rho_ethane, g, length_front, surface_tension)
    numbers[substrate]["front"]["capillary_number"] = capillary_number(surface_tension, velocity, dynamic_viscosity)
    numbers[substrate]["front"]["weber_number"] = weber_number(rho_ethane, velocity, length_front, surface_tension)
    numbers[substrate]["front"]["reynolds_number"] = reynolds_number(rho_ethane, velocity, length_front, dynamic_viscosity)
    numbers[substrate]["front"]["ohnesorge_number"] = ohnesorge_number(dynamic_viscosity, rho_ethane, surface_tension, length_front)
    numbers[substrate]["front"]["froude_number"] = froude_number(velocity, length_front, g)
    numbers[substrate]["front"]["froude_number_root"] = froude_number_root(velocity, length_front, g)
    numbers[substrate]["front"]["characterstic_length_mm"] = length_front*1000  # Characteristic length for front view
    numbers[substrate]["front"]["capillary_length"] = capillary_length(surface_tension, rho_ethane, g)  # Capillary length in meters
    # Side view calculations
    length_side = reference_side_view_dimension[substrate]
    numbers[substrate]["side"]["bond_number"] = bond_number(rho_ethane, g, length_side, surface_tension)
    numbers[substrate]["side"]["capillary_number"] = capillary_number(surface_tension, velocity, dynamic_viscosity)
    numbers[substrate]["side"]["weber_number"] = weber_number(rho_ethane, velocity, length_side, surface_tension)
    numbers[substrate]["side"]["reynolds_number"] = reynolds_number(rho_ethane, velocity, length_side, dynamic_viscosity)
    numbers[substrate]["side"]["ohnesorge_number"] = ohnesorge_number(dynamic_viscosity, rho_ethane, surface_tension, length_side)
    numbers[substrate]["side"]["froude_number"] = froude_number(velocity, length_side, g)
    numbers[substrate]["side"]["froude_number_root"] = froude_number_root(velocity, length_side, g)
    numbers[substrate]["side"]["characterstic_length_mm"] = length_side*1000  # Characteristic length for side view
    numbers[substrate]["side"]["capillary_length"] = capillary_length(surface_tension, rho_ethane, g)  # Capillary length in meters



In [5]:
numbers_df = []
for substrate in numbers:
    for view in numbers[substrate]:
        for name_number, value in numbers[substrate][view].items():
            numbers_df.append({
                "substrate": substrate,
                "view": view,
                "number": name_number,
                "value": value
            })
            

In [6]:
import pandas as pd
numbers_df = pd.DataFrame(numbers_df)
numbers_df.head()

/tmp/ipykernel_17681/1410156165.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


,substrate,view,number,value
0,cryochip,front,bond_number,0.215635
1,cryochip,front,capillary_number,0.007046
2,cryochip,front,weber_number,21.981190
3,cryochip,front,reynolds_number,3119.756086
4,cryochip,front,ohnesorge_number,0.001503


In [7]:
test_substrate = "cryoemgrid"
# show non-dimensional numbers for a specific substrate front and side view 
numbers_for_test_substrate = numbers_df[numbers_df["substrate"] == test_substrate]
# add pivot so that views are in columns and numbers are in rows
numbers_for_test_substrate_pivot = numbers_for_test_substrate.pivot(index="number", columns="view", values="value")
numbers_for_test_substrate_pivot


view,front,side
number,,
bond_number,0.485180,0.000034
capillary_length,0.002153,0.002153
capillary_number,0.007046,0.007046
characterstic_length_mm,1.500000,0.012500
froude_number,67.957866,8154.943935
froude_number_root,8.243656,90.304728
ohnesorge_number,0.001227,0.013442
reynolds_number,4679.634129,38.996951
weber_number,32.971785,0.274765


In [17]:
# Other fluid properties 

mesh_height = 25e-6 # m 
mesh_width = 50e-6 # m
curvature_pressure = surface_tension * mesh_height / mesh_width**2 
print(f"Curvature pressure for mesh: {curvature_pressure:.4f} Pa")

Curvature pressure for mesh: 250.0000 Pa


In [22]:
depth = 1.5e-3 # m
angle_of_exposure_degrees = 60
angle_of_exposure_radians = np.radians(angle_of_exposure_degrees)
impregnation_pressure = rho_ethane * velocity**2 * np.power(np.cos(angle_of_exposure_radians), 2) + rho_ethane * g * depth
print(f"Impregnation pressure: {impregnation_pressure:.4f} Pa")

Impregnation pressure: 145.4688 Pa


In [19]:
wetting_depth_hydrostatic = surface_tension * mesh_height / (rho_ethane * g * mesh_width**2)
print(f"Wetting depth hydrostatic: {wetting_depth_hydrostatic*1000:.4f} mm")

Wetting depth hydrostatic: 46.3746 mm


In [15]:
# shallow seal pinch off time calculations
# https://www.cambridge.org/core/services/aop-cambridge-core/content/view/99FBC492E968BDCEE9CC8AC8DA98A3CD/S0022112008004382a.pdf/water-entry-of-small-hydrophobic-spheres.pdf
# 6.12 equation
test_substrate = "autogrid"
front_characteristic_length = numbers[test_substrate]["front"]["characterstic_length_mm"] * 1e-3  # Convert mm to m
side_characteristic_length = numbers[test_substrate]["side"]["characterstic_length_mm"] * 1e-3  # Convert mm to m
weber_number_front = numbers[test_substrate]["front"]["weber_number"]
weber_number_side = numbers[test_substrate]["side"]["weber_number"]
t_pinch_front = np.sqrt(rho_ethane * front_characteristic_length**3 / surface_tension)
t_pinch_side = np.sqrt(rho_ethane * side_characteristic_length**3 / surface_tension)
print(f"Pinch-off time for front view: {t_pinch_front*1000:.2f} ms")
print(f"Pinch-off time for side view: {t_pinch_side*1000:.4f} ms")
rayleigh_plateau_time = np.sqrt(weber_number_front)
rayleigh_plateau_time_side = np.sqrt(weber_number_side)
print(f"Rayleigh-Plateau time for front view: {rayleigh_plateau_time:.4f} s")
print(f"Rayleigh-Plateau time for side view: {rayleigh_plateau_time_side:.4f} s")


Pinch-off time for front view: 10.85 ms
Pinch-off time for side view: 0.2724 ms
Rayleigh-Plateau time for front view: 6.2022 s
Rayleigh-Plateau time for side view: 1.8158 s


In [16]:
thickness_list = [900e-6, 300e-6, 100e-6, 25e-6]
# Calculate Weber number for different thicknesses
print(f"Using material properties for ethane at {T_C:.2f} °C")
print(f"Velocity: {velocity} m/s")
print(f"Surface tension: {surface_tension} N/m")
print(f"Density: {rho_ethane} kg/m^3")
for thickness in thickness_list:
    weber_number_thickness = weber_number(rho_ethane, velocity, thickness, surface_tension)
    #print(f"Weber number for thickness {thickness*1e6:.0f} microns: {weber_number_thickness:.2f}")
    bond_number_thickness = bond_number(rho_ethane, g, thickness, surface_tension)
    print(f"Bond number for thickness {thickness*1e6:.0f} microns: {bond_number_thickness:.2f}")

Using material properties for ethane at -183.15 °C
Velocity: 1 m/s
Surface tension: 0.025 N/m
Density: 549.5297488543639 kg/m^3
Bond number for thickness 900 microns: 0.17
Bond number for thickness 300 microns: 0.02
Bond number for thickness 100 microns: 0.00
Bond number for thickness 25 microns: 0.00
